# Stage 3 — Blind Expert Evaluation: Statistical Analysis
**Purpose**: Compute exact binomial p-values for individual reviewer preferences (correctness and clarity),
plus inter-rater agreement (Fleiss' kappa) and confidence correlation.

**Input**: `evaluators_responses.txt` (CSV with header)

**Fills placeholders in**: Section 3.3.2 of the manuscript

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import binomtest
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
import warnings
warnings.filterwarnings('ignore')

## 1. Load and clean data

In [2]:
DATA_PATH = r"C:\Users\Admin\OneDrive - Université Libre de Bruxelles\Leo\Insta\UMONS\svet paper\evaluators_responses.txt"

# The file has two header rows (a duplicate header mid-file); read, drop duplicates
df = pd.read_csv(DATA_PATH, sep=',')
# Drop any row where 'reviewer' literally equals the string 'reviewer' (the mid-file duplicate header)
df = df[df['reviewer'] != 'reviewer'].reset_index(drop=True)

# Tidy column names
df.columns = df.columns.str.strip()
df['part'] = df['part'].astype(int)
df['question_number'] = df['question_number'].astype(int)
df['scientific_clarity'] = df['scientific_clarity'].str.strip()
df['factual_correctness'] = df['factual_correctness'].str.strip()
df['confidence'] = df['confidence'].str.strip()
df['rag_plus_pos'] = df['rag_plus_pos'].str.strip()

print(f"Rows loaded: {len(df)}")
print(df['reviewer'].value_counts())

Rows loaded: 147
reviewer
PI (Admin)               42
Guillaume Henderson      42
Alexandre Bastos         21
Sviatlana Lamaka         21
Yaiza Gonzalez Garcia    21
Name: count, dtype: int64


## 2. Resolve preferences to RAG+/RAG−/Tie

For each row, the reviewer chose X, Y, or Tie.  
We compare that choice to `rag_plus_pos` (which position held RAG+) to get the RAG-relative direction.

In [3]:
def resolve(choice, rag_plus_pos):
    """Map reviewer choice (X/Y/Tie) to RAG+/RAG−/Tie."""
    if choice == 'Tie':
        return 'Tie'
    elif choice == rag_plus_pos:
        return 'RAG+'
    else:
        return 'RAG-'

df['clarity_resolved']     = df.apply(lambda r: resolve(r['scientific_clarity'],  r['rag_plus_pos']), axis=1)
df['correctness_resolved'] = df.apply(lambda r: resolve(r['factual_correctness'], r['rag_plus_pos']), axis=1)

df[['reviewer','part','question_number','rag_plus_pos',
    'scientific_clarity','clarity_resolved',
    'factual_correctness','correctness_resolved']].head(10)

,reviewer,part,question_number,rag_plus_pos,scientific_clarity,clarity_resolved,factual_correctness,correctness_resolved
0,Alexandre Bastos,1,1,X,Tie,Tie,Tie,Tie
1,Alexandre Bastos,1,2,Y,Tie,Tie,Tie,Tie
2,Alexandre Bastos,1,3,Y,Y,RAG+,Y,RAG+
3,Alexandre Bastos,1,4,X,Tie,Tie,Tie,Tie
4,Alexandre Bastos,1,5,X,X,RAG+,X,RAG+
5,Alexandre Bastos,1,6,X,X,RAG+,X,RAG+
6,Alexandre Bastos,1,7,Y,X,RAG-,X,RAG-
7,Alexandre Bastos,1,8,Y,Y,RAG+,Y,RAG+
8,Alexandre Bastos,1,9,Y,Tie,Tie,Tie,Tie
9,Alexandre Bastos,1,10,X,X,RAG+,X,RAG+


## 3. Per-reviewer binomial tests

**Null hypothesis**: p(RAG+) = 0.5 among non-Tie judgments (one-sided, alternative = 'greater').

Run separately for **factual correctness** and **scientific clarity**.

In [5]:
# Exclude PI from pooled external stats (keep for separate reporting)
EXTERNAL_REVIEWERS = ['Alexandre Bastos', 'Sviatlana Lamaka', 'Yaiza Gonzalez Garcia', 'Guillaume Henderson']
ALL_REVIEWERS      = EXTERNAL_REVIEWERS + ['PI (Admin)']

def binomial_stats(df_reviewer, col):
    """Return (n_rag_plus, n_rag_minus, n_tie, n_nontie, p_value) for one reviewer and one dimension."""
    n_plus  = (df_reviewer[col] == 'RAG+').sum()
    n_minus = (df_reviewer[col] == 'RAG-').sum()
    n_tie   = (df_reviewer[col] == 'Tie').sum()
    n_non   = n_plus + n_minus
    if n_non == 0:
        return n_plus, n_minus, n_tie, n_non, np.nan
    result  = binomtest(int(n_plus), int(n_non), p=0.5, alternative='greater')
    return int(n_plus), int(n_minus), int(n_tie), int(n_non), result.pvalue

rows = []
for reviewer in ALL_REVIEWERS:
    sub = df[df['reviewer'] == reviewer]
    for dim, col in [('Factual Correctness', 'correctness_resolved'),
                     ('Scientific Clarity',  'clarity_resolved')]:
        n_plus, n_minus, n_tie, n_non, pval = binomial_stats(sub, col)
        rows.append({
            'Reviewer':  reviewer,
            'Dimension': dim,
            'N_questions': len(sub),
            'RAG+': n_plus,
            'RAG-': n_minus,
            'Tie':  n_tie,
            'Non-tie': n_non,
            'RAG+ %': f"{100*n_plus/n_non:.1f}%" if n_non > 0 else 'N/A',
            'p-value (one-sided)': round(pval, 4) if not np.isnan(pval) else 'N/A',
            'Significant (α=0.05)': 'YES' if (not np.isnan(pval) and pval < 0.05) else 'no'
        })

results = pd.DataFrame(rows)
print(results.to_string(index=False))

             Reviewer           Dimension  N_questions  RAG+  RAG-  Tie  Non-tie RAG+ %  p-value (one-sided) Significant (α=0.05)
     Alexandre Bastos Factual Correctness           21    13     3    5       16  81.2%               0.0106                  YES
     Alexandre Bastos  Scientific Clarity           21    13     3    5       16  81.2%               0.0106                  YES
     Sviatlana Lamaka Factual Correctness           21    11     4    6       15  73.3%               0.0592                   no
     Sviatlana Lamaka  Scientific Clarity           21    12     4    5       16  75.0%               0.0384                  YES
Yaiza Gonzalez Garcia Factual Correctness           21     8    10    3       18  44.4%               0.7597                   no
Yaiza Gonzalez Garcia  Scientific Clarity           21     8    10    3       18  44.4%               0.7597                   no
  Guillaume Henderson Factual Correctness           42     6     6   30       12  50.0%   

## 4. Pooled external reviewers (excluding PI)

These are the pooled figures reported in Section 3.3.1 overview.

In [6]:
df_ext = df[df['reviewer'].isin(EXTERNAL_REVIEWERS)]

for dim, col in [('Factual Correctness', 'correctness_resolved'),
                 ('Scientific Clarity',  'clarity_resolved')]:
    n_plus  = (df_ext[col] == 'RAG+').sum()
    n_minus = (df_ext[col] == 'RAG-').sum()
    n_tie   = (df_ext[col] == 'Tie').sum()
    n_non   = n_plus + n_minus
    result  = binomtest(int(n_plus), int(n_non), p=0.5, alternative='greater')
    print(f"\n[POOLED EXTERNAL] {dim}")
    print(f"  RAG+ wins: {n_plus} | RAG- wins: {n_minus} | Ties: {n_tie} | Non-ties: {n_non}")
    print(f"  RAG+ share of non-ties: {100*n_plus/n_non:.1f}%")
    print(f"  Binomial p (one-sided, H1: p>0.5): {result.pvalue:.4e}")


[POOLED EXTERNAL] Factual Correctness
  RAG+ wins: 38 | RAG- wins: 23 | Ties: 44 | Non-ties: 61
  RAG+ share of non-ties: 62.3%
  Binomial p (one-sided, H1: p>0.5): 3.6089e-02

[POOLED EXTERNAL] Scientific Clarity
  RAG+ wins: 48 | RAG- wins: 35 | Ties: 22 | Non-ties: 83
  RAG+ share of non-ties: 57.8%
  Binomial p (one-sided, H1: p>0.5): 9.3730e-02


## 5. Fleiss' kappa — inter-rater agreement

Computed only on questions where ≥ 2 external reviewers rated the same question.  
Note: Bastos (Part 1 only) and Lamaka (Part 2 only) each cover 21 different questions;
Gonzalez Garcia covers Part 1; Henderson covers both parts.

**Overlapping questions per part**:
- Part 1: Bastos + Gonzalez Garcia + Henderson (3 raters, 21 questions)
- Part 2: Lamaka + Bonin (2 raters, 21 questions)

In [7]:
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters

# Encode RAG+ = 2, Tie = 1, RAG- = 0
encode = {'RAG+': 2, 'Tie': 1, 'RAG-': 0}

def compute_fleiss(df_sub, col, raters):
    """
    df_sub: subset of df with only the relevant raters and part
    col: 'correctness_resolved' or 'clarity_resolved'
    raters: list of reviewer names
    """
    pivot = df_sub[df_sub['reviewer'].isin(raters)].pivot_table(
        index='question_number', columns='reviewer', values=col, aggfunc='first'
    ).dropna()  # keep only questions answered by all raters
    
    if pivot.empty or pivot.shape[0] < 2:
        return None, 0
    
    # Encode
    encoded = pivot.applymap(lambda x: encode.get(x, np.nan))
    encoded = encoded.dropna()
    
    # aggregate_raters expects (n_subjects, n_raters) array of category labels
    table, _ = aggregate_raters(encoded.values.astype(int), n_cat=3)
    kappa = fleiss_kappa(table, method='fleiss')
    return round(kappa, 3), len(encoded)

print("Fleiss' kappa — Factual Correctness")
for part, raters in [
    (1, ['Alexandre Bastos', 'Yaiza Gonzalez Garcia', 'Guillaume Henderson']),
    (2, ['Sviatlana Lamaka', 'Guillaume Henderson'])
]:
    df_part = df[df['part'] == part]
    k, n = compute_fleiss(df_part, 'correctness_resolved', raters)
    print(f"  Part {part} ({', '.join(raters)}): κ = {k}  (n={n} questions)")

print("\nFleiss' kappa — Scientific Clarity")
for part, raters in [
    (1, ['Alexandre Bastos', 'Yaiza Gonzalez Garcia', 'Guillaume Henderson']),
    (2, ['Sviatlana Lamaka', 'Guillaume Henderson'])
]:
    df_part = df[df['part'] == part]
    k, n = compute_fleiss(df_part, 'clarity_resolved', raters)
    print(f"  Part {part} ({', '.join(raters)}): κ = {k}  (n={n} questions)")

from sklearn.metrics import cohen_kappa_score

print("\nCohen's kappa — Part 2 (Lamaka vs Bonin)")
df_p2 = df[df['part'] == 2]

for col, label in [('correctness_resolved', 'Factual Correctness'),
                   ('clarity_resolved',     'Scientific Clarity')]:
    pivot = df_p2[df_p2['reviewer'].isin(['Sviatlana Lamaka', 'Guillaume Henderson'])].pivot_table(
        index='question_number', columns='reviewer', values=col, aggfunc='first'
    ).dropna()

    if len(pivot) < 2:
        print(f"  {label}: insufficient data")
        continue

    lamaka = pivot['Sviatlana Lamaka'].values
    bonin  = pivot['Guillaume Henderson'].values
    k = cohen_kappa_score(lamaka, bonin)
    print(f"  {label}: κ = {round(k, 3)}  (n={len(pivot)} questions)")

Fleiss' kappa — Factual Correctness
  Part 1 (Alexandre Bastos, Yaiza Gonzalez Garcia, Guillaume Henderson): κ = -0.069  (n=21 questions)
  Part 2 (Sviatlana Lamaka, Guillaume Henderson): κ = 0.087  (n=21 questions)

Fleiss' kappa — Scientific Clarity
  Part 1 (Alexandre Bastos, Yaiza Gonzalez Garcia, Guillaume Henderson): κ = 0.005  (n=21 questions)
  Part 2 (Sviatlana Lamaka, Guillaume Henderson): κ = 0.095  (n=21 questions)

Cohen's kappa — Part 2 (Lamaka vs Bonin)
  Factual Correctness: κ = 0.182  (n=21 questions)
  Scientific Clarity: κ = 0.154  (n=21 questions)


## 6. Confidence correlation

Do reviewers who are more confident also show a stronger directional preference?
Encode confidence as Low=1, Medium=2, High=3 and compute point-biserial correlation
with a binary RAG+ preference indicator (1 = chose RAG+, 0 = chose RAG- or Tie).

In [8]:
from scipy.stats import pointbiserialr, spearmanr

conf_map = {'low': 1, 'medium': 2, 'high': 3}
df['conf_num'] = df['confidence'].str.lower().map(conf_map)

# Binary: 1 if chose RAG+, 0 otherwise (including Ties)
df['correct_binary'] = (df['correctness_resolved'] == 'RAG+').astype(int)
df['clarity_binary']  = (df['clarity_resolved']     == 'RAG+').astype(int)

print("Spearman correlation: confidence vs. RAG+ preference")
for reviewer in EXTERNAL_REVIEWERS:
    sub = df[df['reviewer'] == reviewer].dropna(subset=['conf_num'])
    for dim, col in [('Correctness', 'correct_binary'), ('Clarity', 'clarity_binary')]:
        r, p = spearmanr(sub['conf_num'], sub[col])
        print(f"  {reviewer:<30} {dim:<15} rho={r:+.3f}  p={p:.3f}")

Spearman correlation: confidence vs. RAG+ preference
  Alexandre Bastos               Correctness     rho=+0.381  p=0.089
  Alexandre Bastos               Clarity         rho=+0.381  p=0.089
  Sviatlana Lamaka               Correctness     rho=+0.309  p=0.173
  Sviatlana Lamaka               Clarity         rho=+0.312  p=0.169
  Yaiza Gonzalez Garcia          Correctness     rho=+0.068  p=0.770
  Yaiza Gonzalez Garcia          Clarity         rho=+0.068  p=0.770
  Guillaume Henderson            Correctness     rho=-0.019  p=0.906
  Guillaume Henderson            Clarity         rho=+0.007  p=0.966


## 7. Summary table for copy-paste into manuscript

This reproduces Table 4 with the exact p-values added.

In [9]:
# Pivot results table for easy reading
summary = results.pivot(index='Reviewer', columns='Dimension', values=[
    'N_questions','RAG+','RAG-','Tie','Non-tie','RAG+ %','p-value (one-sided)','Significant (α=0.05)'
])
print(summary.to_string())

                              N_questions                                   RAG+                                   RAG-                                    Tie                                Non-tie                                 RAG+ %                    p-value (one-sided)                    Significant (α=0.05)                   
Dimension             Factual Correctness Scientific Clarity Factual Correctness Scientific Clarity Factual Correctness Scientific Clarity Factual Correctness Scientific Clarity Factual Correctness Scientific Clarity Factual Correctness Scientific Clarity Factual Correctness Scientific Clarity  Factual Correctness Scientific Clarity
Reviewer                                                                                                                                                                                                                                                                                                                                  